### Sum Rules Evaluation

We proceed to calculate the sum rules using quad for all trained models.

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.integrate import quad, IntegrationWarning

warnings.simplefilter("ignore", IntegrationWarning)


# -------------------------
# Custom Keras layers
# -------------------------


@tf.keras.utils.register_keras_serializable()
class InputScaling(tf.keras.layers.Layer):
    """Logarithmic scaling of the input data."""

    def call(self, inputs):
        x = inputs[:, 0:1]
        q2 = inputs[:, 1:2]

        log_x = tf.math.log(x + 1e-10)
        log_q2 = tf.math.log(q2 + 1e-10)

        return tf.concat([x, log_x, log_q2], axis=-1)


@tf.keras.utils.register_keras_serializable()
class Preprocessing(tf.keras.layers.Layer):
    def __init__(self, noutput, **kwargs):
        super().__init__(**kwargs)
        self.noutput = noutput

    def build(self, input_shape):
        self._beta = self.add_weight(
            shape=(self.noutput,),
            initializer=tf.keras.initializers.Constant(1.0),
            trainable=True,
            name="beta",
            constraint=tf.keras.constraints.non_neg(),
        )

    def call(self, inputs):
        x = inputs[:, 0:1]
        return (1.0 - x) ** (self._beta + 1.0)

    def get_config(self):
        config = super().get_config()
        config.update({"noutput": self.noutput})
        return config


# -------------------------
# Physical constants and parameters
# -------------------------

Q = 91.19
Q2 = np.float32(Q**2)

XMIN = 1e-5
XMAX = 1.0

TMIN = np.log(XMIN)
TMAX = np.log(XMAX)

OUTPUT_BASIS = [-4, -3, -2, -1, 21, 1, 2, 3, 4]
PID_INDEX = {pid: i for i, pid in enumerate(OUTPUT_BASIS)}

THEORETICAL_VALUES = {
    "Momentum": 1.0,
    "u valence": 2.0,
    "d valence": 1.0,
    "s valence": 0.0,
    "c valence": 0.0,
}


# -------------------------
# Models configurations
# -------------------------

MODEL_CONFIGS = [
    {
        "name": "Main Model",
        "architecture": "Model1",
        "model_path": "outputs/main_model.keras",
        "pdf_path": "data/pdf_targets.npy",
        "pids_path": "data/pids_info.npy",
    },
    {
        "name": "Model 1",
        "architecture": "Model1",
        "model_path": "outputs/multimodel/model1.keras",
        "pdf_path": "data/pdf_targets_training.npy",
        "pids_path": "data/pids_info_training.npy",
    },
    {
        "name": "Model 2",
        "architecture": "Model1",
        "model_path": "outputs/multimodel/model2.keras",
        "pdf_path": "data/pdf_targets_training.npy",
        "pids_path": "data/pids_info_training.npy",
    },
    {
        "name": "Model 3",
        "architecture": "Model2",
        "model_path": "outputs/multimodel/model3.keras",
        "pdf_path": "data/pdf_targets_training.npy",
        "pids_path": "data/pids_info_training.npy",
    },
    {
        "name": "Model 4",
        "architecture": "Model3",
        "model_path": "outputs/multimodel/model4.keras",
        "pdf_path": "data/pdf_targets_training.npy",
        "pids_path": "data/pids_info_training.npy",
    },
    {
        "name": "Model 5",
        "architecture": "Model4",
        "model_path": "outputs/multimodel/model5.keras",
        "pdf_path": "data/pdf_targets_training.npy",
        "pids_path": "data/pids_info_training.npy",
    },
    {
        "name": "Model 6",
        "architecture": "Model5",
        "model_path": "outputs/multimodel/model6.keras",
        "pdf_path": "data/pdf_targets_training.npy",
        "pids_path": "data/pids_info_training.npy",
    },
    {
        "name": "Model 7",
        "architecture": "Model6",
        "model_path": "outputs/multimodel/model7.keras",
        "pdf_path": "data/pdf_targets_training.npy",
        "pids_path": "data/pids_info_training.npy",
    },
]


# -------------------------
# Auxiliary functions
# -------------------------


def compute_max_vals(pdf_path, pids_path):
    val_pdf = np.load(pdf_path)
    pids = list(np.load(pids_path))
    pids = [int(pid) for pid in pids]

    pid_cols = {pid: i for i, pid in enumerate(pids)}

    missing_pids = [pid for pid in OUTPUT_BASIS if pid not in pid_cols]
    if missing_pids:
        raise ValueError(f"Missing PIDs in {pids_path}: {missing_pids}")

    output_data = np.zeros((len(val_pdf), len(OUTPUT_BASIS)), dtype=np.float64)

    for j, pid in enumerate(OUTPUT_BASIS):
        output_data[:, j] = val_pdf[:, pid_cols[pid]]

    max_vals = np.max(output_data, axis=0)

    return tf.constant(max_vals, dtype=tf.float32)


def load_pdf_model(model_path):
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            "InputScaling": InputScaling,
            "Preprocessing": Preprocessing,
        },
        compile=False,
    )

    return model


def make_xfx_vector_function(model, max_vals_tf):
    def xfx_vector(x):
        entrada = tf.constant([[x, Q2]], dtype=tf.float32)

        y_norm = model(entrada, training=False)[0]
        y_phys = y_norm * max_vals_tf

        return y_phys.numpy().astype(np.float64)

    return xfx_vector


def compute_sum_rules(model, max_vals_tf):
    """
    Calculate the summation rules using logarithmic substitution:

    x = exp(t)
    dx = x dt

    This is usually more stable than integrating directly with respect to x.
    """

    xfx_vector = make_xfx_vector_function(model, max_vals_tf)

    def momentum_integrand_logx(t):
        x = np.exp(t)
        y = xfx_vector(x)

        return float(np.sum(y) * x)

    def u_valence_integrand_logx(t):
        x = np.exp(t)
        y = xfx_vector(x)

        xu = y[PID_INDEX[2]]
        xubar = y[PID_INDEX[-2]]

        return float(xu - xubar)

    def d_valence_integrand_logx(t):
        x = np.exp(t)
        y = xfx_vector(x)

        xd = y[PID_INDEX[1]]
        xdbar = y[PID_INDEX[-1]]

        return float(xd - xdbar)

    def s_valence_integrand_logx(t):
        x = np.exp(t)
        y = xfx_vector(x)

        xs = y[PID_INDEX[3]]
        xsbar = y[PID_INDEX[-3]]

        return float(xs - xsbar)

    def c_valence_integrand_logx(t):
        x = np.exp(t)
        y = xfx_vector(x)

        xc = y[PID_INDEX[4]]
        xcbar = y[PID_INDEX[-4]]

        return float(xc - xcbar)

    quad_kwargs = {
        "limit": 200,
        "epsabs": 1e-6,
        "epsrel": 1e-6,
    }

    momentum, _ = quad(momentum_integrand_logx, TMIN, TMAX, **quad_kwargs)
    u_valence, _ = quad(u_valence_integrand_logx, TMIN, TMAX, **quad_kwargs)
    d_valence, _ = quad(d_valence_integrand_logx, TMIN, TMAX, **quad_kwargs)
    s_valence, _ = quad(s_valence_integrand_logx, TMIN, TMAX, **quad_kwargs)
    c_valence, _ = quad(c_valence_integrand_logx, TMIN, TMAX, **quad_kwargs)

    return {
        "Momentum": momentum,
        "u valence": u_valence,
        "d valence": d_valence,
        "s valence": s_valence,
        "c valence": c_valence,
    }


# -------------------------
# Results evaluation
# -------------------------

model_results = {}

for cfg in MODEL_CONFIGS:
    print(f"Evaluating {cfg['name']} ({cfg['architecture']})...")

    model_path = cfg["model_path"]
    pdf_path = cfg["pdf_path"]
    pids_path = cfg["pids_path"]

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")

    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Target PDF file not found: {pdf_path}")

    if not os.path.exists(pids_path):
        raise FileNotFoundError(f"Target PIDs file not found: {pids_path}")

    model = load_pdf_model(model_path)
    max_vals_tf = compute_max_vals(pdf_path, pids_path)

    sum_rules = compute_sum_rules(model, max_vals_tf)

    model_results[cfg["name"]] = sum_rules


# -------------------------
# Final results table
# -------------------------

rows = []

for rule_name, theoretical_value in THEORETICAL_VALUES.items():
    row = {
        "Sum Rule": rule_name,
        "Theoretical": theoretical_value,
    }

    for cfg in MODEL_CONFIGS:
        model_name = cfg["name"]
        row[model_name] = model_results[model_name][rule_name]

    rows.append(row)


df_sum_rules = pd.DataFrame(rows)

display(df_sum_rules.style.format(precision=6))

print()
print(f"Q  = {Q} GeV")

Evaluating Main Model (Model1)...
Evaluating Model 1 (Model1)...
Evaluating Model 2 (Model1)...
Evaluating Model 3 (Model2)...
Evaluating Model 4 (Model3)...
Evaluating Model 5 (Model4)...
Evaluating Model 6 (Model5)...
Evaluating Model 7 (Model6)...


,Sum Rule,Theoretical,Main Model,Model 1,Model 2,Model 3,Model 4,Model 5,Model 6,Model 7
0,Momentum,1.000000,0.970416,1.034560,1.003936,1.047227,0.991037,1.068393,0.644392,1.041411
1,u valence,2.000000,1.966457,1.944737,1.871857,1.962249,1.948384,2.128444,2.492665,1.863831
2,d valence,1.000000,0.959049,0.873838,0.946067,1.039366,0.955243,1.182800,0.458333,0.955770
3,s valence,0.000000,0.017489,0.013274,-0.003712,0.055923,0.021427,0.023968,-0.012471,0.008822
4,c valence,0.000000,-0.001069,-0.004106,-0.060049,0.000831,0.001793,0.114091,0.001616,0.012272



Q  = 91.19 GeV
